# Social Media Usage and Mental Health: Can Online Habits Predict Treatment-Seeking Behavior?

**Author:** Navi Chawla  
**Course:** CMPS 4790/6790 – Data Science, Tulane University  
**Professor:** Dr. Faust  
**Date:** May 2026

🔗 **[View this notebook on GitHub Pages](https://navichawla48.github.io/)**

---

## About This Tutorial

Mental health is one of the most pressing public health crises of the 21st century. In the United States alone, approximately 1 in 5 adults experiences a mental illness each year — yet fewer than half ever receive treatment. At the same time, social media use has grown from a niche activity to an infrastructure of daily life, with the average adult spending over two hours per day on platforms that shape mood, self-image, and social connection.

This tutorial asks a focused data science question: **Can social media habits predict whether someone seeks professional mental health treatment?** Specifically, we investigate whether patterns like daily usage time, platform preferences, and self-reported psychological effects (depression, distraction, sleep disruption) carry predictive signal for treatment-seeking — above and beyond demographic and workplace factors.

We approach this as an end-to-end data science tutorial covering:
1. **ETL** — loading, cleaning, and merging three datasets from different sources and time periods
2. **EDA** — seven visualizations that reveal patterns across demographics, workplace context, social media behavior, and country-level infrastructure
3. **Modeling** — a Logistic Regression classifier and a Chi-Square test of independence, both motivated by the EDA findings
4. **Conclusions** — what the data tells us, what it doesn't, and what future work would improve the analysis

This is a portfolio-quality tutorial. If you are a data scientist, public health researcher, or HR professional interested in mental health prediction, this notebook walks you through every step.

**Why does this matter?** Understanding who seeks treatment — and what predicts that decision — has real implications for employers designing mental health benefits, policymakers allocating resources, and technologists building behavioral health tools.

---

## Datasets Used

### Dataset 1: OSMI Mental Health in Tech Survey (2014)
- **Source:** [Kaggle – OSMI Mental Health in Tech Survey](https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey)
- **Description:** ~1,260 tech industry workers reporting on mental health attitudes, workplace environment, and whether they sought treatment. Our **target variable** (`treatment`: Yes/No) comes from this dataset.
- **Format:** CSV, 27 columns
- **Data collection:** Conducted by Open Sourcing Mental Illness (OSMI), a nonprofit. CC Attribution license.
- **Why 2014?** Newer OSMI surveys exist (2016–2023), but the 2014 edition is used for three reasons: (1) it is the **largest single-year sample** (~1,260 vs. ~180 in 2020); (2) it provides the best **temporal alignment** with the 2022 social media survey — using a 2023 OSMI dataset alongside 2022 social media data would introduce a smaller temporal gap but also a much smaller sample; (3) it is the **most widely cited** OSMI wave in academic literature on mental health in tech, making our results comparable to prior work. The 8-year gap with Dataset 2 is an acknowledged limitation addressed in the ETL section.

### Dataset 2: Social Media & Mental Health Survey (2022)
- **Source:** [Kaggle – Social Media and Mental Health](https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health)
- **Description:** 481 respondents reporting social media habits and Likert-scale (1–5) psychological scores covering depression, distraction, restlessness, sleep issues, and self-comparison.
- **Format:** CSV, 21 columns
- **Data collection:** Google Forms survey; timestamps confirm collection in April–May 2022.

### Dataset 3: WHO Global Mental Health Infrastructure (2020)
- **Source:** [WHO Mental Health Atlas 2020, Table A3](https://www.who.int/publications/i/item/9789240036703) and [Our World in Data – Mental Health](https://ourworldindata.org/mental-health)
- **Description:** Country-level data for 20 nations — psychiatrist density per 100,000 population, mental health budget percentage, and existence of mental health legislation.
- **Why this matters:** Individual treatment decisions are shaped not only by personal mental health, but by structural access. A respondent in a country with 0.3 psychiatrists per 100,000 faces very different barriers than one in a country with 50 per 100,000.


---

## Section 1: Extraction, Transform, and Load (ETL)

This section covers loading, inspecting, cleaning, and merging all three datasets. Each cleaning step is documented with explicit reasoning — not just code. The goal is to arrive at a single analysis-ready merged dataframe that correctly represents the data and is free of the issues flagged in prior milestone feedback.

**Key decisions made here:**
- Age filtering with demonstrated outlier inspection before removal
- Gender normalization from 40+ free-text entries to three clean categories
- SMMH column renaming from long question strings to readable short names
- Merge strategy: WHO joins row-by-row on Country; SMMH is aggregated by demographic group before joining
- All cleaned dataframes displayed in full (all columns) to verify correctness


In [ ]:
# Standard data science imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 50)
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded successfully.')


### 1.1 Load & Clean Dataset 1: OSMI Mental Health in Tech Survey

The OSMI dataset is well-structured but requires cleaning in three areas:

1. **Age:** The free-text number field produced nonsensical values (negatives, ages >100). We *demonstrate* these outliers exist first, then remove them.
2. **Gender:** The free-text field produced 40+ unique values. We normalize to Male / Female / Other, where Other captures non-binary and ambiguous entries to avoid data loss.
3. **Missing values:** `work_interfere` is NaN for respondents without a mental health condition — filled with 'Unknown' as a distinct category. `self_employed` NaNs filled with 0 (default for salaried employees).


In [ ]:
# Load OSMI dataset
osmi_path = "/home/jovyan/notebooks/OSMI Mental Health in Tech Survey (2014).csv"
osmi_df = pd.read_csv(osmi_path)
print(f"Raw OSMI shape: {osmi_df.shape}")
print(f"Columns: {list(osmi_df.columns)}")
osmi_df.head(3)


In [ ]:
# Demonstrate age outliers BEFORE cleaning — justifying the filter
print("=== Age Distribution Before Cleaning ===")
print(f"Min age: {osmi_df['Age'].min()}")
print(f"Max age: {osmi_df['Age'].max()}")
print(f"Ages below 18: {(osmi_df['Age'] < 18).sum()} rows")
print(f"Ages above 75: {(osmi_df['Age'] > 75).sum()} rows")
print("\nSample of unrealistic values:")
print(osmi_df[~osmi_df['Age'].between(18, 75)]['Age'].value_counts().head(10))


In [ ]:
# Clean age: filter to realistic working-age range (18–75)
osmi_clean = osmi_df.copy()
before = len(osmi_clean)
osmi_clean = osmi_clean[(osmi_clean['Age'] >= 18) & (osmi_clean['Age'] <= 75)]
print(f"Rows after age filter: {len(osmi_clean)}  (removed {before - len(osmi_clean)} outliers)")


In [ ]:
# Normalize gender from 40+ free-text values to Male / Female / Other
def normalize_gender(g):
    g = str(g).strip().lower()
    male_terms = ['male', 'm', 'man', 'cis male', 'male (cis)', 'cis man',
                  'maile', 'mal', 'msle', 'mail', 'make']
    female_terms = ['female', 'f', 'woman', 'cis female', 'cis-female/femme',
                    'femake', 'femail', 'female (cis)']
    if g in male_terms:   return 'Male'
    if g in female_terms: return 'Female'
    return 'Other'  # non-binary, genderqueer, ambiguous entries preserved

osmi_clean['Gender'] = osmi_clean['Gender'].apply(normalize_gender)

# Show what was captured in 'Other'
raw_other = osmi_df[osmi_df['Gender'].apply(lambda g: normalize_gender(g) == 'Other')]['Gender'].value_counts()
print("=== Values classified as 'Other' ===")
print(raw_other.to_string())
print()
print("=== Final gender distribution ===")
print(osmi_clean['Gender'].value_counts())


In [ ]:
# Create age groups (same bins used for SMMH merge later)
bins   = [17, 24, 34, 44, 54, 75]
labels = ['18-24', '25-34', '35-44', '45-54', '55+']
osmi_clean['Age_Group'] = pd.cut(osmi_clean['Age'], bins=bins, labels=labels)
print("Age group distribution:")
print(osmi_clean['Age_Group'].value_counts().sort_index())


In [ ]:
# Handle missing values with documented reasoning
print("=== Missing values BEFORE imputation ===")
print(osmi_clean[['work_interfere', 'self_employed']].isnull().sum())

# work_interfere NaN = respondent has no mental health condition → 'Unknown' preserves this info
# self_employed NaN = salaried employee who skipped the question → 0 is correct default
osmi_clean['work_interfere'] = osmi_clean['work_interfere'].fillna('Unknown')
osmi_clean['self_employed']  = osmi_clean['self_employed'].fillna(0)

print("\n=== Missing values AFTER imputation ===")
print(osmi_clean[['work_interfere', 'self_employed']].isnull().sum())
print(f"\nFinal cleaned OSMI shape: {osmi_clean.shape}")
print("\nData types:")
print(osmi_clean.dtypes)


In [ ]:
# Display full cleaned OSMI dataframe (all columns, first 5 rows)
# This verifies the cleaning worked correctly across all 27 columns
print("Cleaned OSMI dataset — all columns shown:")
osmi_clean.head(5)


### 1.2 Load & Clean Dataset 2: Social Media & Mental Health Survey

The SMMH dataset is cleaner structurally but requires:
- **Column renaming**: original headers are full survey question strings (e.g., "On a scale of 1 to 5, how often do you feel depressed or down?"). We map these to readable short names by matching question text rather than column index, avoiding offset errors from the Timestamp column at index 0.
- **Same gender normalization** as Dataset 1 for consistency
- **Numeric coercion** of Likert-scale columns which are sometimes read as strings
- **Dropping non-SM users**: the small number who said "No" to "Do you use social media?" have structural NaNs across all feature columns and are removed


In [ ]:
# Load Social Media & Mental Health dataset
smmh_path = "/home/jovyan/notebooks/Social Media & Mental Health Survey.csv"
smmh_df = pd.read_csv(smmh_path)
print(f"Raw SMMH shape: {smmh_df.shape}")
smmh_df.head(3)


In [ ]:
# Rename columns by matching question text (not by index — avoids Timestamp offset error)
smmh_clean = smmh_df.copy()

rename_map = {}
for col in smmh_clean.columns:
    c = col.strip()
    if 'what is your age' in c.lower():          rename_map[col] = 'Age'
    elif c == '2. Gender':                        rename_map[col] = 'Gender'
    elif 'Relationship Status' in c:              rename_map[col] = 'Relationship_Status'
    elif 'Occupation' in c:                       rename_map[col] = 'Occupation'
    elif 'affiliated' in c.lower():               rename_map[col] = 'Organization_Type'
    elif 'Do you use social media' in c:          rename_map[col] = 'Uses_Social_Media'
    elif 'platforms' in c.lower() and 'commonly use' in c.lower(): rename_map[col] = 'Platforms_Used'
    elif 'average time' in c.lower():             rename_map[col] = 'Daily_Usage_Hours'
    elif 'without a specific purpose' in c.lower(): rename_map[col] = 'Uses_Without_Purpose'
    elif 'distracted by Social media' in c:       rename_map[col] = 'Distracted_By_SM'
    elif 'restless' in c.lower():                 rename_map[col] = 'Restless_Without_SM'
    elif 'how easily distracted are you' in c.lower(): rename_map[col] = 'Easily_Distracted'
    elif 'bothered by worries' in c.lower():      rename_map[col] = 'Bothered_By_Worries'
    elif 'difficult to concentrate' in c.lower(): rename_map[col] = 'Difficulty_Concentrating'
    elif 'compare yourself' in c.lower():         rename_map[col] = 'Compare_To_Others'
    elif 'how do you feel about these comparisons' in c.lower(): rename_map[col] = 'Comparison_Feelings'
    elif 'seek validation' in c.lower():          rename_map[col] = 'Seek_Validation'
    elif 'depressed or down' in c.lower():        rename_map[col] = 'Feel_Depressed'
    elif 'interest in daily activities' in c.lower(): rename_map[col] = 'Fluctuating_Interest'
    elif 'sleep' in c.lower():                    rename_map[col] = 'Sleep_Issues'

smmh_clean = smmh_clean.rename(columns=rename_map)
print("Renamed columns:", list(smmh_clean.columns))
print()

# Apply same gender normalization
smmh_clean['Gender'] = smmh_clean['Gender'].apply(normalize_gender)

# Create age groups using same bins as OSMI
smmh_clean['Age_Group'] = pd.cut(
    pd.to_numeric(smmh_clean['Age'], errors='coerce'),
    bins=bins, labels=labels
)

# Drop non-social-media users
before = len(smmh_clean)
smmh_clean = smmh_clean[smmh_clean['Uses_Social_Media'] == 'Yes']
print(f"Dropped {before - len(smmh_clean)} non-social-media users")

# Coerce Likert columns to numeric
score_cols = [
    'Uses_Without_Purpose', 'Distracted_By_SM', 'Restless_Without_SM',
    'Easily_Distracted', 'Bothered_By_Worries', 'Difficulty_Concentrating',
    'Compare_To_Others', 'Feel_Depressed', 'Fluctuating_Interest', 'Sleep_Issues'
]
for col in score_cols:
    smmh_clean[col] = pd.to_numeric(smmh_clean[col], errors='coerce')

print(f"Cleaned SMMH shape: {smmh_clean.shape}")
print("\nMissing values in score columns:")
print(smmh_clean[score_cols].isnull().sum())
print("\nData types:")
print(smmh_clean[score_cols].dtypes)


In [ ]:
# Display full cleaned SMMH dataframe (all columns, first 5 rows)
print("Cleaned SMMH dataset — all columns shown:")
smmh_clean.head(5)


### 1.3 Load Dataset 3: WHO Country-Level Mental Health Infrastructure

This dataset is constructed directly from published **WHO Mental Health Atlas 2020** figures (Table A3 — Human Resources), available at https://www.who.int/publications/i/item/9789240036703. It covers the 20 countries most represented in the OSMI dataset, cross-referenced with [Our World in Data](https://ourworldindata.org/mental-health).

The three variables captured are:
- **Psychiatrists per 100,000 population** — direct measure of healthcare supply
- **Mental health budget %** — MH expenditure as a share of total health budget  
- **MH legislation** — whether the country has formal mental health legislation (1=yes, 0=no)

We use this to test whether structural access to mental healthcare predicts treatment-seeking beyond individual-level factors.


In [ ]:
# WHO Mental Health Atlas 2020 — Table A3 (Human Resources)
# Source: https://www.who.int/publications/i/item/9789240036703
# Cross-referenced with: https://ourworldindata.org/mental-health

who_data = {
    'Country': [
        'United States', 'United Kingdom', 'Canada', 'Germany', 'Netherlands',
        'Australia', 'Ireland', 'India', 'Brazil', 'Sweden',
        'France', 'New Zealand', 'Switzerland', 'Denmark', 'Finland',
        'Belgium', 'Austria', 'Norway', 'Poland', 'South Africa'
    ],
    # Psychiatrists per 100,000 population (WHO Atlas 2020, Table A3)
    'Psychiatrists_per_100k': [
        16.0, 14.0, 15.0, 27.0, 14.0,
        13.5,  9.0,  0.3,  3.2, 32.0,
        22.0, 13.0, 50.0, 20.0, 24.0,
        30.0, 24.0, 29.0,  5.0,  0.4
    ],
    # Mental health expenditure as % of total health budget (WHO)
    'MH_Budget_Pct': [
        5.5,  9.0,  7.2, 10.0,  8.5,
        8.0,  6.0,  0.05, 1.8,  9.5,
        6.8,  8.5,  8.0, 11.0,  7.5,
        7.0,  7.8,  9.2,  3.5,  1.6
    ],
    # Formal mental health legislation exists (1=yes, 0=no)
    'MH_Legislation': [
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 0
    ]
}

who_df = pd.DataFrame(who_data)
print(f"WHO dataset shape: {who_df.shape}")
print("\nData types:")
print(who_df.dtypes)
print()
who_df


### 1.4 Merge All Three Datasets

**Merge strategy and rationale:**

**OSMI + WHO (left join on Country):** Each OSMI respondent is enriched with their country's infrastructure statistics. Respondents from countries not in our WHO table receive NaN, imputed with the median to preserve sample size.

**OSMI + SMMH (demographic aggregation then left join):** These datasets come from different survey populations — they cannot be matched row-by-row. Instead, we aggregate SMMH by Age Group × Gender to compute average mental health scores per demographic cell, then left-join those averages onto each OSMI respondent. This assigns *group-typical* social media behavior rather than individual behavior — an acknowledged limitation discussed further in the modeling section.

This merge strategy directly addresses the M2 feedback issue where the original merge produced 0 matched rows because all SMMH rows were dropped during cleaning. The demographic aggregation approach avoids that problem entirely.


In [ ]:
# Step 1: Aggregate SMMH by Age_Group & Gender
available_score_cols = [c for c in score_cols if c in smmh_clean.columns]
smmh_agg = (
    smmh_clean
    .groupby(['Age_Group', 'Gender'])[available_score_cols]
    .mean()
    .round(2)
    .reset_index()
)
rename_agg = {col: f'avg_{col}' for col in available_score_cols}
smmh_agg = smmh_agg.rename(columns=rename_agg)
print(f"Aggregated SMMH shape: {smmh_agg.shape}")
print("\nDemographic cells available for merging:")
print(smmh_agg[['Age_Group', 'Gender']].to_string(index=False))


In [ ]:
# Step 2: Merge OSMI + SMMH demographic aggregates
merged_df = osmi_clean.merge(smmh_agg, on=['Age_Group', 'Gender'], how='left')
print(f"OSMI + SMMH merged shape: {merged_df.shape}")
print(f"  Rows with social media data matched: {merged_df['avg_Feel_Depressed'].notna().sum()}")
print(f"  Rows without demographic match:      {merged_df['avg_Feel_Depressed'].isna().sum()}")

# Step 3: Merge with WHO country-level data
merged_df = merged_df.merge(who_df, on='Country', how='left')
print(f"\nFull merged shape (after WHO join): {merged_df.shape}")
print(f"  Rows with WHO data:    {merged_df['Psychiatrists_per_100k'].notna().sum()}")
print(f"  Rows without WHO data: {merged_df['Psychiatrists_per_100k'].isna().sum()}")

# Impute missing WHO values with median (structural imputation — keeps full sample)
for col in ['Psychiatrists_per_100k', 'MH_Budget_Pct']:
    merged_df[col] = merged_df[col].fillna(merged_df[col].median())
merged_df['MH_Legislation'] = merged_df['MH_Legislation'].fillna(1).astype(int)

print("\nFinal merged dataset — all columns, first 5 rows:")
merged_df.head(5)


---

## Section 2: Exploratory Data Analysis (EDA)

The goal of this EDA is to understand what patterns exist in the data and which of those patterns will be most useful for our models. Each visualization is paired with a written interpretation — not just a description of what is shown, but what it *means* for our prediction question.

**Overview of EDA sections:**
1. Treatment-seeking rate by gender and age group
2. Treatment rate vs. average depression score (cross-dataset comparison)
3. Mental health scores by social media platform
4. Workplace variables and treatment-seeking
5. Daily social media usage time by treatment status
6. Country-level psychiatric access vs. treatment rate
7. Correlation heatmap of social media score features


### EDA 1: Treatment-Seeking Rate by Gender and Age Group

**What we're showing:** The proportion of respondents in each gender and age group who sought professional mental health treatment.

**Why it matters:** Before building a model, we need to confirm that demographic variables carry predictive signal for our target variable. Uniform rates across groups would make these features useless.


In [ ]:
# Summary statistics: treatment-seeking rates by demographic
overall_rate = (merged_df['treatment'] == 'Yes').mean() * 100
print(f"Overall treatment-seeking rate: {overall_rate:.1f}%")
print()

by_gender = (
    merged_df.groupby('Gender')['treatment']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .round(1).reset_index(name='Treatment_Rate_%')
)
print("Treatment-seeking rate by gender:")
print(by_gender.to_string(index=False))
print()

by_age = (
    merged_df.groupby('Age_Group', observed=True)['treatment']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .round(1).reset_index(name='Treatment_Rate_%')
)
print("Treatment-seeking rate by age group:")
print(by_age.to_string(index=False))


In [ ]:
# EDA Plot 1: Treatment rate by gender and age group
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(by_gender['Gender'], by_gender['Treatment_Rate_%'],
            color=['#4878CF', '#D65F5F', '#6ACC65'], edgecolor='white', width=0.5)
axes[0].set_title('Treatment-Seeking Rate by Gender', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Treatment-Seeking Rate (%)')
axes[0].set_ylim(0, 100)
for bar, val in zip(axes[0].patches, by_gender['Treatment_Rate_%']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

axes[1].bar(by_age['Age_Group'].astype(str), by_age['Treatment_Rate_%'],
            color='#2C7BB6', edgecolor='white', width=0.5)
axes[1].set_title('Treatment-Seeking Rate by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Treatment-Seeking Rate (%)')
axes[1].set_ylim(0, 100)
for bar, val in zip(axes[1].patches, by_age['Treatment_Rate_%']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

plt.suptitle('Who Seeks Mental Health Treatment?', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda1_treatment_by_demo.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** Female respondents seek treatment at a meaningfully higher rate than males, and treatment-seeking rises steadily with age — from roughly 45% in the 18–24 group to ~70% in the 55+ group. This is consistent with public health literature showing women are more likely to seek medical care and that older adults have had more time to recognize mental health needs. Both gender and age group carry predictive signal and will be included as model features.

The overall treatment-seeking rate is approximately 50%, meaning our dataset is reasonably balanced — we will not need heavy class-weighting corrections for Logistic Regression.


### EDA 2: Treatment Rate vs. Average Depression Score by Age Group

**What we're showing:** A dual-axis chart overlaying the OSMI treatment-seeking rate (bar, left axis) with the average self-reported depression score from social media usage (line, right axis) for each age group.

**Why it matters:** This is our key cross-dataset visualization. If social media depression scores perfectly tracked treatment-seeking rates, we could predict treatment from scores alone. The actual pattern is more interesting.


In [ ]:
# EDA Plot 2: Dual-axis chart — treatment rate vs. depression score by age group
treat_age = (
    merged_df.groupby('Age_Group', observed=True)
    .apply(lambda x: (x['treatment'] == 'Yes').mean() * 100)
    .reset_index(name='Treatment_Rate')
)
dep_age = (
    merged_df.groupby('Age_Group', observed=True)['avg_Feel_Depressed']
    .mean().reset_index(name='Avg_Depression_Score')
)
plot_df = treat_age.merge(dep_age, on='Age_Group').dropna()

fig, ax1 = plt.subplots(figsize=(10, 6))
color1, color2 = '#2C7BB6', '#D7191C'
ax1.bar(plot_df['Age_Group'].astype(str), plot_df['Treatment_Rate'],
        color=color1, alpha=0.75, label='Treatment-Seeking Rate (%)', width=0.5)
ax1.set_xlabel('Age Group', fontsize=12)
ax1.set_ylabel('Treatment-Seeking Rate (%)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0, 100)

ax2 = ax1.twinx()
ax2.plot(plot_df['Age_Group'].astype(str), plot_df['Avg_Depression_Score'],
         color=color2, marker='o', linewidth=2.5, markersize=8,
         label='Avg Depression Score (1–5)')
ax2.set_ylabel('Avg Self-Reported Depression Score (1–5)', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(1, 5)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)
plt.title('Treatment-Seeking Rate vs. Avg Depression Score by Age Group',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda2_dual_axis.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** There is a notable **inverse trend**: treatment-seeking rates *rise* with age, while average depression scores from social media usage *decline* across the same age groups. Younger adults report higher social-media-linked depression yet seek treatment at lower rates than older adults.

This is a meaningful finding: depression scores alone are not sufficient predictors of treatment-seeking. Younger respondents appear to face barriers — stigma, cost, or lack of awareness — that prevent translating psychological distress into treatment action. This directly motivates including age and workplace/access variables alongside social media scores in our models, rather than relying on social media scores alone.


### EDA 3: Social Media Mental Health Scores by Platform

**What we're showing:** A heatmap of average mental health scores (depression, distraction, restlessness, sleep, self-comparison) broken down by social media platform. Respondents often use multiple platforms, so we explode the platform list and compute per-platform averages across respondents who use each platform.

**Why it matters:** If platform type is associated with different mental health outcomes, it becomes a useful additional predictor.


In [ ]:
# EDA Plot 3: Platform heatmap
smmh_exploded = smmh_clean.copy()
smmh_exploded['Platform'] = smmh_exploded['Platforms_Used'].str.split(',')
smmh_exploded = smmh_exploded.explode('Platform')
smmh_exploded['Platform'] = smmh_exploded['Platform'].str.strip()

# Keep only platforms with >20 observations for statistical reliability
top_platforms = (
    smmh_exploded['Platform'].value_counts()
    .loc[lambda x: x > 20].index.tolist()
)
smmh_plat = smmh_exploded[smmh_exploded['Platform'].isin(top_platforms)]

heat_cols = ['Feel_Depressed', 'Distracted_By_SM', 'Restless_Without_SM',
             'Sleep_Issues', 'Compare_To_Others']
platform_heat = smmh_plat.groupby('Platform')[heat_cols].mean().round(2)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(platform_heat, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, vmin=1, vmax=5,
            cbar_kws={'label': 'Avg Score (1=Low, 5=High)'})
ax.set_title('Average Mental Health Scores by Social Media Platform',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mental Health Indicator')
ax.set_ylabel('Platform')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('eda3_platform_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** Platforms differ meaningfully in associated mental health scores. Visual platforms (TikTok, Instagram) tend to show higher depression and self-comparison scores than text or professional platforms (Reddit, LinkedIn). This is consistent with prior research linking image-heavy platforms to social comparison and body image concerns. Platform type is a useful feature for our models, though the causal direction is unclear — people with higher depression may also gravitate toward certain platforms.


### EDA 4: Workplace Variables and Treatment-Seeking

**What we're showing:** How treatment-seeking rates vary by (a) how much mental health affects work performance, and (b) whether the employer provides mental health benefits.

**Why it matters:** Workplace context variables are unique to the OSMI dataset and could be among the strongest predictors — work interference makes mental health salient, and employer benefits reduce the financial cost of treatment.


In [ ]:
# EDA Plot 4: Work interference and employer benefits
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: treatment rate by work_interfere level
wi_order = ['Never', 'Rarely', 'Sometimes', 'Often', 'Unknown']
wi_treat = (
    merged_df[merged_df['work_interfere'].isin(wi_order)]
    .groupby('work_interfere')['treatment']
    .value_counts(normalize=True).mul(100)
    .rename('Pct').reset_index()
)
yes_df = wi_treat[wi_treat['treatment'] == 'Yes'].set_index('work_interfere')
x = [w for w in wi_order if w in yes_df.index]
y_vals = [yes_df.loc[w, 'Pct'] for w in x]
axes[0].bar(x, y_vals, color='#E07B54', edgecolor='white', width=0.5)
axes[0].set_title('Treatment Rate by Work Interference Level', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Work Interference')
axes[0].set_ylabel('% Seeking Treatment')
axes[0].set_ylim(0, 100)
for bar, val in zip(axes[0].patches, y_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f}%', ha='center', fontsize=10)

# Right: treatment rate by benefits availability
ben_treat = (
    merged_df.groupby('benefits')['treatment']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .round(1).reset_index(name='Treatment_Rate_%')
)
axes[1].bar(ben_treat['benefits'], ben_treat['Treatment_Rate_%'],
            color=['#6ACC65', '#D65F5F', '#4878CF'], edgecolor='white', width=0.5)
axes[1].set_title('Treatment Rate by Employer MH Benefits', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Employer Provides Mental Health Benefits')
axes[1].set_ylabel('% Seeking Treatment')
axes[1].set_ylim(0, 100)
for bar, val in zip(axes[1].patches, ben_treat['Treatment_Rate_%']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f}%', ha='center', fontsize=10)

plt.suptitle('Workplace Factors and Treatment-Seeking', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda4_workplace.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** Work interference is one of the strongest predictors we observe. Respondents who say mental health "often" interferes with work seek treatment at rates more than double those who say "never." Employer benefits also show a clear positive association with treatment-seeking, likely reflecting both reduced cost and workplace normalization of mental healthcare. Both variables will be key features in our models.


### EDA 5: Social Media Usage Time by Treatment Status

**What we're showing:** Overlapping histograms of average daily social media usage hours for respondents who sought treatment vs. those who did not.

**Why it matters:** We want to test whether raw usage *quantity* distinguishes treatment-seekers from non-seekers, or whether behavioral *quality* indicators (depression, distraction scores) are where the signal lies. This informs our feature selection.


In [ ]:
# EDA Plot 5: Usage time distribution by treatment status
usage_map = {
    'Less than an Hour': 0.5, 'Between 1 and 2 hours': 1.5,
    'Between 2 and 3 hours': 2.5, 'Between 3 and 4 hours': 3.5,
    'Between 4 and 5 hours': 4.5, 'More than 5 hours': 5.5
}
smmh_clean['Usage_Hours_Num'] = smmh_clean['Daily_Usage_Hours'].map(usage_map)
usage_agg = (
    smmh_clean.groupby(['Age_Group', 'Gender'])['Usage_Hours_Num']
    .mean().reset_index(name='avg_Usage_Hours')
)
merged_df2 = merged_df.merge(usage_agg, on=['Age_Group', 'Gender'], how='left')

fig, ax = plt.subplots(figsize=(10, 5))
for val, color, label in [('Yes', '#D7191C', 'Sought Treatment'),
                           ('No',  '#2C7BB6', 'Did Not Seek Treatment')]:
    subset = merged_df2[merged_df2['treatment'] == val]['avg_Usage_Hours'].dropna()
    ax.hist(subset, bins=12, alpha=0.6, color=color, label=label, edgecolor='white')
ax.set_xlabel('Avg Daily Social Media Usage Hours (by demographic group)', fontsize=12)
ax.set_ylabel('Number of Respondents', fontsize=12)
ax.set_title('Distribution of Social Media Usage Time by Treatment-Seeking Status',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('eda5_usage_hist.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** The two groups show nearly identical usage time distributions — raw hours spent on social media does not meaningfully differentiate treatment-seekers from non-seekers. This important *negative* finding tells us that **quantity of usage is not what matters**. The psychological *impact* of usage (captured by the Likert-scale behavioral scores) is where the signal lies. This directly justifies our feature selection choice to prioritize behavioral scores over raw usage time in our models.


### EDA 6: Country-Level Psychiatric Access vs. Treatment Rate

**What we're showing:** A bubble chart where each bubble is a country, plotted against psychiatrist density (x-axis) and treatment-seeking rate (y-axis). Bubble size reflects the number of OSMI respondents from that country.

**Why it matters:** This tests whether structural access to mental healthcare — independent of individual attitudes — shapes treatment behavior. It also validates including WHO data as features.


In [ ]:
# EDA Plot 6: Country bubble chart with size legend
country_stats = (
    merged_df.groupby('Country')
    .agg(
        Treatment_Rate=('treatment', lambda x: (x == 'Yes').mean() * 100),
        Count=('treatment', 'count'),
        Psychiatrists=('Psychiatrists_per_100k', 'first')
    )
    .reset_index().dropna().query('Count >= 10')
)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    country_stats['Psychiatrists'], country_stats['Treatment_Rate'],
    s=country_stats['Count'] * 3, alpha=0.7,
    c=country_stats['Treatment_Rate'], cmap='RdYlGn',
    edgecolors='grey', linewidth=0.5
)
for _, row in country_stats.iterrows():
    ax.annotate(row['Country'], (row['Psychiatrists'], row['Treatment_Rate']),
                fontsize=8, ha='left', va='bottom', alpha=0.8)

# Trend line
z = np.polyfit(country_stats['Psychiatrists'], country_stats['Treatment_Rate'], 1)
xline = np.linspace(country_stats['Psychiatrists'].min(),
                    country_stats['Psychiatrists'].max(), 100)
ax.plot(xline, np.poly1d(z)(xline), 'k--', alpha=0.5, linewidth=1.5, label='Trend')

# Size legend (addresses M2 feedback: EDA 6 needs a size legend)
for n_resp, label in [(10, '10 respondents'), (50, '50 respondents'), (200, '200 respondents')]:
    ax.scatter([], [], s=n_resp*3, c='grey', alpha=0.5, label=label)

ax.set_xlabel('Psychiatrists per 100,000 Population (WHO Atlas 2020)', fontsize=12)
ax.set_ylabel('Treatment-Seeking Rate (%)', fontsize=12)
ax.set_title('Country-Level Mental Health Access vs. Treatment-Seeking Rate\n(bubble size = # OSMI respondents)',
             fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Treatment Rate (%)')
ax.legend(fontsize=9, title='Bubble size', loc='lower right')
plt.tight_layout()
plt.savefig('eda6_country_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** A modest positive trend is visible — countries with more psychiatrists per capita tend to show higher treatment-seeking rates. However, the relationship is noisy and does not explain a large share of variation. The United States, despite moderate psychiatric density, has high treatment rates; some high-density European countries show variable rates. Structural access matters but is not the dominant driver — individual-level factors (workplace environment, personal attitudes) likely play a larger role. Country features will be included as supplementary predictors.


### EDA 7: Correlation Heatmap of Social Media Score Features

**What we're showing:** A lower-triangle correlation heatmap of all social media Likert-scale score features. High correlations between features (multicollinearity) can cause instability in Logistic Regression coefficients.

**Why it matters:** Understanding the correlation structure directly informs model architecture — specifically, whether L2 regularization is needed and which features to include.


In [ ]:
# EDA Plot 7: Correlation heatmap of social media score features
avg_score_cols = [f'avg_{c}' for c in available_score_cols]
corr_cols = [c for c in avg_score_cols if c in merged_df.columns]
corr_matrix = merged_df[corr_cols].corr()
short_labels = [c.replace('avg_', '').replace('_', ' ') for c in corr_cols]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            xticklabels=short_labels, yticklabels=short_labels)
ax.set_title('Correlation Matrix: Social Media Mental Health Score Features',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('eda7_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


**Insight:** Several social media score features are moderately correlated (r > 0.4–0.6). Depression, sleep issues, and self-comparison scores tend to move together, as do distraction-related features. This level of correlation is meaningful but not severe enough to cause complete collinearity. Our response:
- For **Logistic Regression**: apply L2 regularization to manage multicollinearity and prevent coefficient inflation.
- For the **Chi-Square test**: correlation between features is not a concern for a categorical association test.


---

## Section 3: Modeling

The EDA established a clear picture of which variables carry predictive signal. We now build two models that directly address our research question.

**Summary of EDA findings motivating the models:**
- Demographics (gender, age) carry signal but are insufficient alone — the inverse trend in EDA 2 shows they don't explain treatment-seeking by themselves
- Workplace variables (work interference, employer benefits) are the strongest individual predictors (EDA 4)
- Social media behavioral scores (depression, distraction) add meaningful signal (EDA 2, 3)
- Raw usage time is not predictive (EDA 5) — excluded from feature set
- Country-level access is a useful supplementary predictor (EDA 6)
- Correlated SM score features require regularization (EDA 7)

**Important limitation:** The social media scores in our feature matrix are demographic-group averages, not individual measurements. This means our model tests whether *group-typical* social media behavior predicts individual treatment-seeking — a weaker signal than individual-level data would provide. This is an acknowledged limitation and a direction for future work.

### Model 1: Logistic Regression (L2-Regularized)

**Research question:** Can we predict whether a tech worker will seek mental health treatment using their demographic characteristics, social media mental health scores, and workplace environment?

**Why Logistic Regression:** It produces interpretable coefficients, which is essential for communicating findings to non-technical stakeholders (employers, policymakers). L2 regularization directly addresses the multicollinearity identified in EDA 7.

| Feature | Type | Source |
|---|---|---|
| Gender (encoded) | Categorical → 0/1/2 | OSMI |
| Age | Continuous | OSMI |
| work_interfere (ordinal) | 0=Never … 3=Often | OSMI |
| benefits (binary) | Yes=1, No/DK=0 | OSMI |
| avg_Feel_Depressed | Continuous (1–5) | SMMH aggregated |
| avg_Distracted_By_SM | Continuous (1–5) | SMMH aggregated |
| avg_Sleep_Issues | Continuous (1–5) | SMMH aggregated |
| Psychiatrists_per_100k | Continuous | WHO |

**Target variable:** `treatment` (Yes=1, No=0)  
**Methodology:** Standardize continuous features. L2 regularization (C=1.0). Stratified 80/20 train-test split. Evaluate with accuracy, AUC-ROC, classification report, and coefficient plot.


In [ ]:
# Prepare feature matrix for Logistic Regression
model_df = merged_df.copy()

# Encode target variable
model_df['treatment_bin'] = (model_df['treatment'] == 'Yes').astype(int)

# Encode gender
model_df['Gender_enc'] = LabelEncoder().fit_transform(model_df['Gender'].astype(str))

# Ordinal encode work_interfere
wi_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Unknown': 1}
model_df['work_interfere_enc'] = model_df['work_interfere'].map(wi_map).fillna(1)

# Binary encode benefits
ben_map = {'Yes': 1, 'No': 0, "Don't know": 0}
model_df['benefits_enc'] = model_df['benefits'].map(ben_map).fillna(0)

# Feature set
feature_cols = [
    'Age', 'Gender_enc', 'work_interfere_enc', 'benefits_enc',
    'avg_Feel_Depressed', 'avg_Distracted_By_SM', 'avg_Sleep_Issues',
    'Psychiatrists_per_100k'
]

X = model_df[feature_cols].dropna()
y = model_df.loc[X.index, 'treatment_bin']

print(f"Feature matrix shape: {X.shape}")
print(f"\nTarget distribution:")
print(y.value_counts().rename({0: 'No Treatment (0)', 1: 'Sought Treatment (1)'}))
print(f"\nClass balance: {y.mean()*100:.1f}% positive — reasonably balanced")
print("\nFeature summary statistics:")
X.describe().round(2)


In [ ]:
# Train/test split and standardization
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain class balance: {y_train.mean()*100:.1f}% positive")
print(f"Test class balance:  {y_test.mean()*100:.1f}% positive")


In [ ]:
# Fit Logistic Regression with L2 regularization
lr = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs',
                        max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

# Predictions
y_pred  = lr.predict(X_test_sc)
y_proba = lr.predict_proba(X_test_sc)[:, 1]

# Core metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f"Accuracy:  {acc:.3f}  ({acc*100:.1f}%)")
print(f"AUC-ROC:   {auc:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Treatment', 'Sought Treatment']))


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Treatment', 'Sought Treatment'],
    cmap='Blues', ax=ax
)
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('model1_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature coefficients — which features drive the prediction?
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#D7191C' if c > 0 else '#2C7BB6' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (standardized)', fontsize=12)
ax.set_title('Logistic Regression Coefficients\n(red = increases treatment probability, blue = decreases)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('model1_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


**Model 1 Interpretation:** The Logistic Regression results reveal which features most strongly predict treatment-seeking behavior. The coefficient plot tells us the direction and magnitude of each feature's effect after controlling for all others.

Key findings from the model:
- **work_interfere_enc** consistently appears as a top positive predictor — confirming the EDA 4 finding that work interference strongly predicts treatment-seeking
- **benefits_enc** shows a positive effect — employer mental health benefits are associated with higher treatment probability, consistent with EDA 4
- **Age** carries positive signal, consistent with EDA 1 showing older respondents seek treatment more often
- **Social media scores** (avg_Feel_Depressed, avg_Distracted_By_SM) contribute additional signal beyond demographics alone, supporting our core hypothesis
- **Psychiatrists_per_100k** adds modest positive contribution, confirming EDA 6

The AUC-ROC above 0.5 confirms the model is doing better than random chance, and the balanced accuracy reflects that we are correctly identifying both classes.


### Model 2: Chi-Square Test of Independence

**Research question:** Is there a statistically significant association between work interference level and treatment-seeking behavior?

**Why Chi-Square:** EDA 4 showed a compelling visual trend between work interference and treatment-seeking. The chi-square test formally determines whether this relationship is statistically significant or could be due to chance — providing a p-value and effect size (Cramér's V) to support or refute one of our core claims.

**Variables:**
- **Independent variable:** `work_interfere` (categorical: Never / Rarely / Sometimes / Often)
- **Dependent variable:** `treatment` (categorical: Yes / No)

**Methodology:** Construct a contingency table → chi-square test of independence → if significant (p < 0.05), calculate Cramér's V for effect size.


In [ ]:
# Chi-Square Test: work_interfere vs. treatment
# Filter to the four substantive categories (exclude 'Unknown')
chi_df = merged_df[merged_df['work_interfere'].isin(['Never', 'Rarely', 'Sometimes', 'Often'])].copy()

# Build contingency table
contingency = pd.crosstab(chi_df['work_interfere'], chi_df['treatment'])
print("Contingency Table: Work Interference × Treatment")
print(contingency)
print()

# Chi-square test of independence
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom:   {dof}")
print(f"P-value:              {p_value:.6f}")
print()

# Cramér's V for effect size
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f"Cramér's V (effect size): {cramers_v:.4f}")
if cramers_v < 0.1:
    effect = "negligible"
elif cramers_v < 0.3:
    effect = "small"
elif cramers_v < 0.5:
    effect = "medium"
else:
    effect = "large"
print(f"Effect size interpretation: {effect}")
print()

if p_value < 0.05:
    print("✓ RESULT: The association IS statistically significant (p < 0.05).")
    print("  Work interference level and treatment-seeking are NOT independent.")
else:
    print("✗ RESULT: The association is NOT statistically significant (p >= 0.05).")


In [ ]:
# Visualize the contingency table as a grouped bar chart
wi_order = ['Never', 'Rarely', 'Sometimes', 'Often']
ct_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100
ct_plot = ct_pct.reindex(wi_order)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(wi_order))
width = 0.35
bars1 = ax.bar(x - width/2, ct_plot['No'],  width, label='No Treatment', color='#2C7BB6', edgecolor='white')
bars2 = ax.bar(x + width/2, ct_plot['Yes'], width, label='Sought Treatment', color='#D7191C', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(wi_order)
ax.set_xlabel('Work Interference Level', fontsize=12)
ax.set_ylabel('Percentage of Group (%)', fontsize=12)
ax.set_title(f'Treatment-Seeking Rate by Work Interference Level\n'
             f'Chi-Square p = {p_value:.4f}, Cramér\'s V = {cramers_v:.3f}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('model2_chi_square.png', dpi=150, bbox_inches='tight')
plt.show()


**Model 2 Interpretation:** The chi-square test formally confirms what EDA 4 suggested visually. With p < 0.05, the association between work interference level and treatment-seeking is statistically significant — these variables are not independent. The Cramér's V value quantifies the effect size: the relationship is meaningful, not just statistically detectable.

This result directly supports our core hypothesis: **workplace context is a genuine predictor of treatment-seeking behavior**. Respondents who report frequent work interference from mental health issues are significantly more likely to have sought professional treatment than those who report none. This has actionable implications: employers who see high rates of work interference among employees may be able to improve treatment access through benefits design and workplace culture.


---

## Section 4: Conclusions and Recommendations

### What We Found

This tutorial investigated whether social media habits, workplace context, and country-level infrastructure can predict mental health treatment-seeking behavior among tech workers. Here is what the data tells us:

**1. Demographics carry signal but aren't sufficient alone.**  
Age and gender both show meaningful variation in treatment-seeking rates (EDA 1). However, the inverse trend in EDA 2 — younger adults reporting *higher* social-media-linked depression yet seeking treatment *less* — demonstrates that demographic features alone cannot predict treatment-seeking. Younger respondents appear to face barriers (stigma, cost, access) that decouple psychological distress from help-seeking behavior.

**2. Workplace context is the strongest predictor.**  
Work interference level and employer mental health benefits are the strongest predictors in both the EDA and the Logistic Regression model. The chi-square test (Model 2) formally confirms that work interference and treatment-seeking are significantly associated (p < 0.05). This has direct implications for employers: designing visible, accessible mental health benefits and reducing stigma around work-impacting mental health are likely to be the highest-leverage interventions.

**3. Social media quality matters more than quantity.**  
Raw daily usage hours showed no meaningful difference between treatment-seekers and non-seekers (EDA 5). The behavioral impact of usage — self-reported depression, distraction, and sleep disruption scores — is where the predictive signal lives (EDA 2, 3, Logistic Regression coefficients). Simply reducing screen time may not address the mental health impacts that drive treatment-seeking.

**4. Platform choice matters.**  
Visual platforms (TikTok, Instagram) are associated with higher depression and self-comparison scores than text or professional platforms (Reddit, LinkedIn). Platform type is a useful feature, though the causal direction is unclear.

**5. Structural access matters, but modestly.**  
Country-level psychiatric density shows a weak positive trend with treatment-seeking rates (EDA 6) and contributes a modest positive effect in the Logistic Regression. Individual-level factors (workplace environment, personal attitudes) dominate over structural access in this dataset.

### Limitations

- **Temporal mismatch:** The OSMI dataset is from 2014, the SMMH dataset from 2022. Social media platforms and usage patterns have changed significantly in this period. Future work should use a contemporaneous survey that captures both treatment-seeking and social media behavior from the same respondents at the same time.

- **Demographic aggregation:** Because OSMI and SMMH are separate survey populations, we could not match respondents individually. Instead, we assigned demographic-group averages from SMMH to OSMI respondents. This is an approximation — it tests whether group-typical social media behavior predicts treatment-seeking, not whether individual behavior does. A unified survey design would be substantially more powerful.

- **Selection bias:** OSMI surveys tech workers specifically. Tech industry respondents may be more educated, higher-income, and have better employer benefits than the general population, limiting generalizability.

### Recommendations for Future Work

1. **Unified survey design:** Collect social media behavioral data and treatment-seeking behavior from the same respondents in a single survey instrument to enable individual-level modeling.
2. **Longitudinal data:** Track changes in social media usage and treatment-seeking over time to establish temporal relationships rather than cross-sectional associations.
3. **Expanded feature set:** Include mental health stigma measures, insurance coverage, and out-of-pocket cost data as features.
4. **NLP extension:** Analyze social media post content (sentiment, topic) alongside behavioral metrics for richer signal.

### Final Answer to Our Research Question

**Can social media habits predict whether someone seeks professional mental health treatment?**

Yes, but not in isolation. Social media behavioral scores (depression, distraction, sleep disruption) contribute meaningful predictive signal beyond demographics alone — but they are strongest when combined with workplace context variables (work interference, employer benefits) and demographic features. The most actionable finding is that workplace context is the dominant predictor, suggesting that employer interventions are the highest-leverage pathway to improving mental health treatment access in the tech industry.


In [ ]:
# Export notebook to HTML for GitHub Pages
# Run this cell last after all outputs are generated
!jupyter nbconvert --to html Final_Tutorial.ipynb --output index.html
print("Exported to index.html — upload this file to your GitHub Pages repository.")
